# Speed up the transformer: CODI and explicit GPT-2 on T4

**Select GPU T4 x2, enable Internet, and Run All.** One T4 is used. No data attachments
or prior results are needed. This separate notebook keeps the released model and
rank-96 head recipe from the previous experiment.

It tries static KV storage, contiguous projection weights, SDPA attention, compiler
fusion, CUDA Graph replay and preallocated token/mask buffers. Explicit decoding also
tries four-token chunks to reduce host EOS checks. Separate INT8/INT4 Triton kernels
are accuracy-gated candidates. A faster result is measured, not assumed.

**Output order:** equal-MAC experiment, isolated transformer-block substeps, candidate
results, then model timing/accuracy. Only the isolated block probe installs module
hooks, and removes them before returning. Main timing and accuracy runs have no hooks.

Before/after means **current versus selected optimized decoder, with the same rank-96
head**. Four model breakdown tables show per-question and per-token/latent-step means.
Dense-head controls additionally show whether transformer optimization makes head
compression more worthwhile. Full GSM8K accuracy runs last and can take longer than fitting.

In [ ]:
import gc,gzip,hashlib,importlib.util,json,logging,os,pathlib,random,subprocess,sys,time,warnings
from contextlib import contextmanager
from urllib.request import urlopen

# Fixed experiment defaults: no configuration is required.
SEED=89
MODES=['codi','explicit_cot']
FIT_QUESTIONS,SELECT_QUESTIONS,RECOVERY_QUESTIONS=1024,256,256
MAX_FIT_STATES,MAX_SELECT_STATES,MAX_RECOVERY_STATES=4096,1024,2048
CLEAN_EPOCHS,RECOVERY_EPOCHS=4,2
DISTILL_BATCH_SIZE=8
COLLECT_BATCH_SIZE=8
RANKS=(32,64,96)
MAX_NEW_TOKENS={'codi':64,'explicit_cot':256}
TIMING_QUESTIONS,TIMING_REPEATS=16,3
OUTPUT_ROOT=pathlib.Path('/kaggle/working/codi_transformer_optimization')
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
BOOTSTRAP_TIMES=[]
@contextmanager
def bootstrap_stage(name):
    start=time.perf_counter()
    try: yield
    finally: BOOTSTRAP_TIMES.append(dict(name=name,cpu_wall_ms=1000*(time.perf_counter()-start)))

# Do not reinstall datasets/pandas/dill or change Kaggle's PyTorch/CUDA.
# GSM8K is read directly from its canonical JSONL files.
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
os.environ.pop('HF_HUB_DISABLE_XET',None)
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT','300')
REPO_URL='https://github.com/0x0shephard/latent-reasoning.git'
BASE_COMMIT='6a8d2e61950c67f012d0a9ba13ec8a70f3a25019'
REPO_DIR='/kaggle/working/latent-reasoning'
setup_log=OUTPUT_ROOT/'setup.log'
def checked(command):
    result=subprocess.run(command,capture_output=True,text=True)
    with setup_log.open('a') as log: log.write(result.stdout+'\n'+result.stderr+'\n')
    if result.returncode:
        raise RuntimeError(result.stdout[-3000:]+'\n'+result.stderr[-5000:])
    return result
with bootstrap_stage('git_clone_and_checkout'):
    if not pathlib.Path(REPO_DIR).exists(): checked(['git','clone',REPO_URL,REPO_DIR])
    checked(['git','-C',REPO_DIR,'fetch','origin'])
    checked(['git','-C',REPO_DIR,'checkout','--detach',BASE_COMMIT])
os.chdir(REPO_DIR)
sys.path.insert(0,REPO_DIR)
with bootstrap_stage('dependency_installation'):
    checked([sys.executable,'-m','pip','install','-q','transformers==4.52.4','peft==0.15.2',
             'huggingface_hub>=0.34.0,<1.0','hf_xet','accelerate==1.7.0','pyyaml'])
    probe=subprocess.run([sys.executable,'-c','import peft; from transformers import GPT2LMHeadModel'],capture_output=True,text=True)
    if probe.returncode and 'torchao' in (probe.stdout+probe.stderr):
        checked([sys.executable,'-m','pip','uninstall','-y','torchao'])
    checked([sys.executable,'-c','import torch,peft,huggingface_hub,hf_xet; from transformers import GPT2LMHeadModel,DynamicCache'])
print('Setup ready. Unrelated system-package messages are retained in setup.log; failed installs/imports stop here.')

In [ ]:
RUNTIME_SOURCE = '"""CODI/explicit decoding and an additive, four-column bottleneck report.\n\nThe report partitions a single CUDA-stream timeline, including host-induced idle\nintervals. It is an instrumented elapsed-time breakdown, not summed kernel time.\n"""\nfrom __future__ import annotations\n\nfrom contextlib import contextmanager\nfrom dataclasses import dataclass\nimport gzip\nimport json\nfrom pathlib import Path\nimport time\nimport uuid\n\nimport torch\nfrom torch import nn\nfrom src.models.official_codi import official_codi_base_model, _normalized_official_questions\nfrom src.inference.official_codi_fast import FastCODIGeneration\n\n\nclass Timeline:\n    def __init__(self, device=\'cpu\', enabled=True, unified_clock=False):\n        self.device = torch.device(device)\n        self.enabled = enabled\n        self.unified_clock = unified_clock\n        self.context = {}\n        self.records = []\n        self.pending = []\n        self.stack = []\n        self.counter = 0\n        self.trace_id = uuid.uuid4().hex\n\n    def start(self, name, kind=\'stage\', gpu=True, **extra):\n        if not self.enabled:\n            return None\n        self.counter += 1\n        row = dict(self.context, trace_id=self.trace_id, event_id=self.counter,\n                   parent_id=self.stack[-1] if self.stack else None,\n                   name=name, kind=kind, **extra)\n        marker = None if self.unified_clock else torch.profiler.record_function(name)\n        if marker is not None:\n            marker.__enter__()\n        row[\'_wall_start\'] = time.perf_counter()\n        events = None\n        if (gpu or self.unified_clock) and self.device.type == \'cuda\':\n            events = (torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True))\n            events[0].record()\n        self.stack.append(self.counter)\n        return row, events, marker\n\n    def stop(self, token):\n        if token is None:\n            return\n        row, events, marker = token\n        if events:\n            events[1].record()\n        row[\'cpu_wall_ms\'] = (time.perf_counter() - row.pop(\'_wall_start\')) * 1000\n        if marker is not None:\n            marker.__exit__(None, None, None)\n        if self.stack and self.stack[-1] == row[\'event_id\']:\n            self.stack.pop()\n        self.pending.append((row, events))\n\n    @contextmanager\n    def span(self, name, kind=\'stage\', gpu=True, **extra):\n        token = self.start(name, kind, gpu, **extra)\n        try:\n            yield\n        finally:\n            self.stop(token)\n\n    @contextmanager\n    def metadata(self, **values):\n        previous = self.context.copy()\n        self.context.update(values)\n        try:\n            yield\n        finally:\n            self.context = previous\n\n    def resolve(self):\n        if self.device.type == \'cuda\' and self.pending:\n            torch.cuda.synchronize(self.device)\n        for row, events in self.pending:\n            row[\'cuda_stream_ms\'] = events[0].elapsed_time(events[1]) if events else None\n            self.records.append(row)\n        self.pending.clear()\n\n    @contextmanager\n    def modules(self, roots, recursive=True):\n        if not self.enabled:\n            yield\n            return\n        handles, stacks, seen = [], {}, set()\n        for prefix, root in roots:\n            for suffix, module in (root.named_modules() if recursive else [(\'\', root)]):\n                if id(module) in seen:\n                    continue\n                seen.add(id(module))\n                name = prefix + (\'.\' + suffix if suffix else \'\')\n                stacks[id(module)] = []\n                def pre(mod, args, label=name):\n                    token = self.start(label, kind=\'module\', inclusive=True,\n                                       module_type=type(mod).__name__)\n                    stacks[id(mod)].append(token)\n                def post(mod, args, output):\n                    self.stop(stacks[id(mod)].pop())\n                handles.append(module.register_forward_pre_hook(pre))\n                handles.append(module.register_forward_hook(post, always_call=True))\n        try:\n            yield\n        finally:\n            for handle in handles:\n                handle.remove()\n\n    def flush(self, path):\n        self.resolve()\n        path = Path(path)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        with gzip.open(path, \'at\', encoding=\'utf-8\') as handle:\n            for row in self.records:\n                handle.write(json.dumps(row, default=str) + \'\\n\')\n        self.records.clear()\n\n\n@dataclass\nclass PreparedBatch:\n    indices: tuple\n    ids: torch.Tensor\n    mask: torch.Tensor\n\n\ndef prepare_questions(tokenizer, questions, batch_size, timeline=None):\n    timeline = timeline or Timeline(enabled=False)\n    with timeline.span(\'question_normalization\', gpu=False):\n        questions = _normalized_official_questions(questions)\n    with timeline.span(\'question_tokenization\', gpu=False):\n        encoded = tokenizer(questions, add_special_tokens=False, padding=False)[\'input_ids\']\n    result = []\n    for start in range(0, len(encoded), batch_size):\n        part = encoded[start:start+batch_size]\n        with timeline.metadata(batch_index=start//batch_size):\n            with timeline.span(\'cpu_padding_and_tensor_allocation\', gpu=False):\n                width = max(map(len, part))\n                ids = torch.full((len(part), width), tokenizer.pad_token_id, dtype=torch.long)\n                mask = torch.zeros_like(ids)\n                for i, values in enumerate(part):\n                    if not values:\n                        raise ValueError(\'Empty question\')\n                    ids[i, -len(values):] = torch.tensor(values)\n                    mask[i, -len(values):] = 1\n            result.append(PreparedBatch(tuple(range(start, start+len(part))), ids, mask))\n    return result\n\n\ndef select_token(head, hidden, vocabulary_stop):\n    specialized = getattr(head, \'select_token\', None)\n    return specialized(hidden, vocabulary_stop=vocabulary_stop) if callable(specialized) else head(hidden)[..., :vocabulary_stop].argmax(-1)\n\n\n@torch.no_grad()\ndef decode(model, tokenizer, batches, head, *, mode, device, max_new_tokens=256,\n           latent_iterations=6, timeline=None, observer=None, forced_tokens=None):\n    """Same CODI body-only contract; explicit mode starts directly from the question.\n\n    Fixed replay evaluates every head but feeds back dense-reference token IDs.\n    Timed CUDA paths must be called inside torch.inference_mode() by the runner.\n    """\n    if mode not in (\'codi\', \'explicit_cot\'):\n        raise ValueError(mode)\n    if max_new_tokens <= 0:\n        raise ValueError(\'max_new_tokens must be positive\')\n    timeline = timeline or Timeline(device, enabled=False)\n    device = torch.device(device)\n    base = official_codi_base_model(model)\n    body, embedding = base.transformer, model.input_embeddings()\n    model.eval()\n    head.eval()\n    vocabulary_stop = int(model.eot_id)\n    eos = int(tokenizer.eos_token_id)\n    total = sum(len(b.indices) for b in batches)\n    outputs, texts, counts_out = [None]*total, [None]*total, [0]*total\n    with timeline.span(\'answer_cue_tokenization\', gpu=False):\n        cue_ids = tokenizer(\' The answer is:\', add_special_tokens=False)[\'input_ids\'] if mode == \'codi\' else []\n    for batch_number, batch in enumerate(batches):\n        with timeline.metadata(batch_index=batch_number, question_indices=list(batch.indices), token_position=-1):\n            with timeline.span(\'host_to_device_input_ids\'):\n                ids = batch.ids.to(device, non_blocking=True)\n            with timeline.span(\'host_to_device_attention_mask\'):\n                mask = batch.mask.to(device, non_blocking=True)\n            with timeline.span(\'prompt_tensor_construction\'):\n                if mode == \'codi\':\n                    bot = torch.full((len(ids),1), model.bot_id, device=device, dtype=torch.long)\n                    ids = torch.cat((ids,bot),1)\n                    mask = torch.cat((mask,torch.ones_like(bot)),1)\n            # Do not silently truncate GPT-2 context.\n            maximum = getattr(model.config, \'n_positions\', 1024) if hasattr(model, \'config\') else 1024\n            reserve = latent_iterations + 1 + len(cue_ids) if mode == \'codi\' else 0\n            if ids.shape[1] + reserve + max_new_tokens > maximum:\n                raise ValueError(\'Prompt plus generation exceeds context; reduce the configured token cap\')\n            with timeline.span(\'prefill_position_ids\'):\n                positions = mask.long().cumsum(-1)-1 if mode == \'explicit_cot\' else None\n                if positions is not None:\n                    positions.masked_fill_(mask == 0, 1)\n            with timeline.metadata(phase=\'prefill\'):\n                with timeline.span(\'transformer_prefill\'):\n                    prefill_kwargs = dict(input_ids=ids, attention_mask=mask, use_cache=True, return_dict=True)\n                    if getattr(getattr(body, \'config\', None), \'model_type\', None) == \'gpt2\':\n                        from transformers import DynamicCache\n                        prefill_kwargs[\'past_key_values\'] = DynamicCache()\n                    if positions is not None:\n                        prefill_kwargs[\'position_ids\'] = positions\n                    out = body(**prefill_kwargs)\n            with timeline.span(\'kv_cache_reference_update\'):\n                cache = out.past_key_values\n                hidden = out.last_hidden_state[:, -1:, :]\n            if mode == \'codi\':\n                with timeline.metadata(phase=\'latent\'), timeline.span(\'phase_latent\'):\n                    with timeline.metadata(phase=\'latent_projection_initial\'):\n                        with timeline.span(\'latent_projector\'):\n                            latent = model.prj(hidden)\n                    for step in range(latent_iterations):\n                        with timeline.metadata(phase=\'latent\', latent_step=step):\n                            with timeline.span(\'transformer_latent_pass\'):\n                                out = body(inputs_embeds=latent, past_key_values=cache, use_cache=True, return_dict=True)\n                            with timeline.span(\'kv_cache_reference_update\'):\n                                cache = out.past_key_values\n                            with timeline.span(\'latent_projector\'):\n                                latent = model.prj(out.last_hidden_state[:, -1:, :])\n            with timeline.metadata(phase=\'visible_decode\'), timeline.span(\'phase_visible\'):\n                if mode == \'codi\':\n                    with timeline.metadata(phase=\'answer_cue\'):\n                        with timeline.span(\'answer_cue_tensor_construction\'):\n                            cue = torch.tensor([model.eot_id,*cue_ids], device=device).unsqueeze(0).expand(len(ids),-1)\n                        with timeline.span(\'answer_cue_embedding\'):\n                            cue_embedding = embedding(cue)\n                        with timeline.span(\'transformer_answer_cue\'):\n                            out = body(inputs_embeds=cue_embedding, past_key_values=cache, use_cache=True, return_dict=True)\n                        cache = out.past_key_values\n                        hidden = out.last_hidden_state[:, -1:, :]\n                with timeline.span(\'generation_buffer_allocation\'):\n                    tokens = torch.full((len(ids), max_new_tokens), eos, device=device, dtype=torch.long)\n                    counts = torch.zeros(len(ids), device=device, dtype=torch.long)\n                    finished = torch.zeros(len(ids), device=device, dtype=torch.bool)\n                replay = None\n                if forced_tokens is not None:\n                    with timeline.span(\'replay_cpu_padding\', gpu=False):\n                        seqs = [forced_tokens[i] for i in batch.indices]\n                        limit = max(map(len,seqs))\n                        if limit > max_new_tokens or any(not seq for seq in seqs):\n                            raise ValueError(\'Replay tokens must fit the generation cap\')\n                        replay_cpu = torch.full((len(ids),limit), eos, dtype=torch.long)\n                        for row, seq in enumerate(seqs):\n                            replay_cpu[row,:len(seq)] = torch.tensor(seq)\n                    with timeline.span(\'host_to_device_replay_tokens\'):\n                        replay = replay_cpu.to(device)\n                else:\n                    limit = max_new_tokens\n                for position in range(limit):\n                    with timeline.metadata(phase=\'visible_decode\', token_position=position):\n                        if observer:\n                            with timeline.span(\'collect_hidden_state_to_cpu\'):\n                                observer(hidden[:, -1, :], ~finished, position, batch.indices)\n                        with timeline.span(\'lm_head_and_argmax\'):\n                            predicted = select_token(head, hidden[:, -1, :], vocabulary_stop)\n                        with timeline.span(\'token_selection_and_buffers\'):\n                            token = predicted if replay is None else replay[:, position]\n                            active = ~finished\n                            tokens[:,position] = torch.where(active,token,tokens[:,position])\n                            counts += active.long()\n                            finished |= active & (token == eos)\n                        with timeline.span(\'termination_check_host_sync\'):\n                            stop = position+1 == limit or bool(finished.all())\n                        if stop:\n                            break\n                        with timeline.span(\'next_token_embedding\'):\n                            embedded = embedding(token).unsqueeze(1)\n                        with timeline.span(\'decode_attention_mask_update\'):\n                            if mode == \'explicit_cot\':\n                                mask = torch.cat((mask,torch.ones((len(ids),1),device=device,dtype=mask.dtype)),1)\n                        with timeline.span(\'transformer_visible_token\'):\n                            kwargs = dict(inputs_embeds=embedded, past_key_values=cache, use_cache=True, return_dict=True)\n                            if mode == \'explicit_cot\':\n                                kwargs[\'attention_mask\'] = mask\n                                kwargs[\'position_ids\'] = (mask.long().sum(-1)-1).unsqueeze(1)\n                            out = body(**kwargs)\n                        with timeline.span(\'kv_cache_reference_update\'):\n                            cache, hidden = out.past_key_values, out.last_hidden_state\n                with timeline.span(\'device_to_host_token_buffer\'):\n                    cpu_tokens = tokens.cpu()\n                with timeline.span(\'device_to_host_counts\'):\n                    cpu_counts = counts.cpu().tolist()\n                with timeline.span(\'cpu_token_conversion_and_text_decode\', gpu=False):\n                    for row,index in enumerate(batch.indices):\n                        count = int(cpu_counts[row])\n                        seq = tuple(int(t) for t in cpu_tokens[row,:count].tolist())\n                        outputs[index], counts_out[index] = seq, count\n                        texts[index] = tokenizer.decode(seq, skip_special_tokens=True)\n            if not timeline.unified_clock:\n                timeline.resolve()\n    return FastCODIGeneration(tuple(texts),tuple(outputs),tuple(counts_out))\n\n\nclass DenseSelector(nn.Module):\n    def __init__(self, head, vocabulary_size):\n        super().__init__()\n        self.head = head\n        self.vocabulary_size = vocabulary_size\n    def forward(self, hidden):\n        return self.head(hidden)[..., :self.vocabulary_size]\n    def select_token(self, hidden, *, vocabulary_stop):\n        return self(hidden).argmax(-1)\n\n\nclass FixedRankHead(nn.Module):\n    def __init__(self, source, rank=96):\n        super().__init__()\n        self.vocabulary_size = source.vocabulary_size\n        self.down = nn.Linear(source.hidden_size,rank)\n        self.up = nn.Linear(rank,source.vocabulary_size)\n        with torch.no_grad():\n            self.down.weight.copy_(source.down.weight[:rank])\n            self.down.bias.copy_(source.down.bias[:rank])\n            self.up.weight.copy_(source.up.weight[:,:rank])\n            self.up.bias.copy_(source.up.bias)\n        self.requires_grad_(False)\n    def forward(self, hidden):\n        return self.up(self.down(hidden))\n    def select_token(self, hidden, *, vocabulary_stop):\n        return self(hidden).argmax(-1)\n\n\n\nCOLUMNS = (\'Explicit\', \'CODI overall\', \'CODI latent only\', \'CODI visible only\')\nBREAKDOWN_ROWS = (\n    \'Question loading / tokenization\', \'CPU padding / allocation\',\n    \'Input transfer to GPU\', \'Embeddings\',\n    *(f\'Transformer block {i + 1:02d}\' for i in range(12)),\n    \'Transformer final norm\', \'Transformer masks / bookkeeping\',\n    \'Latent projector\', \'LM head\', \'Argmax / head dispatch\',\n    \'Token buffers / cache updates\', \'EOS check / synchronization\',\n    \'Output transfer to CPU\', \'Text decoding\', \'Python / tracing gaps\',\n)\n\n\ndef _category(row, ancestors):\n    # Descendant events inherit their enclosing component. Summing exclusive\n    # durations reconstructs that component without counting nested modules twice.\n    for event in [row, *ancestors]:\n        name = event[\'name\']\n        if name.startswith(\'transformer.h.\'):\n            return f"Transformer block {int(name.split(\'.\')[2]) + 1:02d}"\n        if name.startswith(\'transformer.ln_f\'):\n            return \'Transformer final norm\'\n        if name.startswith((\'transformer.wte\', \'transformer.wpe\')):\n            return \'Embeddings\'\n        if name.startswith(\'projector\'):\n            return \'Latent projector\'\n        if name.startswith(\'lm_head\') and name != \'lm_head_and_argmax\':\n            return \'LM head\'\n    name = row[\'name\']\n    if name in (\'load_question\', \'question_normalization\', \'question_tokenization\', \'answer_cue_tokenization\'):\n        return \'Question loading / tokenization\'\n    if name == \'cpu_padding_and_tensor_allocation\':\n        return \'CPU padding / allocation\'\n    if name.startswith(\'host_to_device\'):\n        return \'Input transfer to GPU\'\n    if name in (\'next_token_embedding\', \'answer_cue_embedding\'):\n        return \'Embeddings\'\n    if name.startswith(\'transformer_\') or name in (\'prefill_position_ids\', \'decode_attention_mask_update\'):\n        return \'Transformer masks / bookkeeping\'\n    if name == \'latent_projector\':\n        return \'Latent projector\'\n    if name == \'lm_head_and_argmax\':\n        return \'Argmax / head dispatch\'\n    if name == \'termination_check_host_sync\':\n        return \'EOS check / synchronization\'\n    if name.startswith(\'device_to_host\'):\n        return \'Output transfer to CPU\'\n    if name == \'cpu_token_conversion_and_text_decode\':\n        return \'Text decoding\'\n    if name in (\'kv_cache_reference_update\', \'generation_buffer_allocation\',\n                \'token_selection_and_buffers\', \'prompt_tensor_construction\', \'answer_cue_tensor_construction\'):\n        return \'Token buffers / cache updates\'\n    return \'Python / tracing gaps\'\n\n\ndef partition_question(events):\n    """Exclusive intervals on one clock; never add CPU and CUDA measurements."""\n    by_id = {r[\'event_id\']: r for r in events}\n    roots = [r for r in events if r[\'name\'] == \'question_total\']\n    if len(roots) != 1:\n        raise ValueError(\'Expected exactly one complete question trace\')\n    root = roots[0]\n    clock = \'cuda_stream_ms\' if root.get(\'cuda_stream_ms\') is not None else \'cpu_wall_ms\'\n    if any(r.get(clock) is None for r in events):\n        raise ValueError(\'Every span must use the same clock; enable unified_clock\')\n    children = {}\n    for row in events:\n        children.setdefault(row.get(\'parent_id\'), []).append(row)\n    values = {}\n    for row in events:\n        exclusive = row[clock] - sum(c[clock] for c in children.get(row[\'event_id\'], []))\n        if exclusive < -0.01:\n            raise ValueError(f"Overlapping timing spans: {row[\'name\']}: {exclusive} ms")\n        # Keep sub-microsecond event rounding differences so totals reconcile.\n        ancestors = []\n        parent = row.get(\'parent_id\')\n        while parent is not None:\n            ancestors.append(by_id[parent]); parent = by_id[parent].get(\'parent_id\')\n        category = _category(row, ancestors)\n        phase = row.get(\'phase\', \'shared\')\n        phase = (\'latent\' if phase in (\'latent\', \'latent_projection_initial\') else\n                 \'visible\' if phase in (\'visible_decode\', \'answer_cue\') else \'shared\')\n        key = (category, phase)\n        values[key] = values.get(key, 0.0) + exclusive\n    return dict(mode=root[\'mode\'], question_id=root.get(\'question_id\'), repeat=root.get(\'repeat\'),\n                total_ms=root[clock], clock=clock,\n                breakdown=[dict(name=name, phase=phase, ms=value) for (name, phase), value in values.items()])\n\n\ndef bottleneck_means(samples, *, per_token=False):\n    """Question means or aggregate time / native generated-step count.\n\n    Token units: explicit = visible output token; CODI overall = latent + visible\n    step; CODI latent = latent step; CODI visible = visible output token. Prompt and\n    forced cue work are amortized, not added to the generated-step denominator.\n    """\n    groups = {mode: [s for s in samples if s[\'mode\'] == mode] for mode in (\'explicit_cot\', \'codi\')}\n    if any(not group for group in groups.values()):\n        raise ValueError(\'Both reasoning modes need timing samples\')\n    rows = {name: dict.fromkeys(COLUMNS, 0.0) for name in BREAKDOWN_ROWS}\n    totals = dict.fromkeys(COLUMNS, 0.0)\n    for mode, group in groups.items():\n        overall = \'Explicit\' if mode == \'explicit_cot\' else \'CODI overall\'\n        if per_token:\n            visible = sum(s[\'visible_tokens\'] for s in group)\n            latent = sum(s[\'latent_steps\'] for s in group)\n            denominators = {overall: visible + latent if mode == \'codi\' else visible,\n                            \'CODI latent only\': latent, \'CODI visible only\': visible}\n            if denominators[overall] <= 0 or (mode == \'codi\' and min(latent, visible) <= 0):\n                raise ValueError(\'Per-token reporting needs positive generated-step counts\')\n        else:\n            denominators = dict.fromkeys(COLUMNS, len(group))\n        for sample in group:\n            totals[overall] += sample[\'total_ms\'] / denominators[overall]\n            for item in sample[\'breakdown\']:\n                value = item[\'ms\'] / denominators[overall]\n                rows[item[\'name\']][overall] += value\n                if mode == \'codi\' and item[\'phase\'] in (\'latent\', \'visible\'):\n                    column = \'CODI latent only\' if item[\'phase\'] == \'latent\' else \'CODI visible only\'\n                    phase_value = item[\'ms\'] / denominators[column]\n                    rows[item[\'name\']][column] += phase_value\n                    totals[column] += phase_value\n    label = \'Total average time per token/step\' if per_token else \'Total average time\'\n    return [(label, totals), *rows.items(), (label + \' (repeat)\', dict(totals))]\n\n\n@torch.inference_mode()\ndef profile_question(model, tokenizer, head, question, *, mode, device, max_new_tokens,\n                     question_id=0, repeat=0, arm=\'rank96\', latent_iterations=6):\n    """One batch-1 sample; only displayed components receive module hooks."""\n    device = torch.device(device)\n    timeline = Timeline(device, unified_clock=True)\n    timeline.context.update(mode=mode, arm=arm, question_id=question_id, repeat=repeat, phase=\'shared\')\n    base = official_codi_base_model(model)\n    if device.type == \'cuda\':\n        torch.cuda.synchronize(device)\n    roots = [(f\'transformer.h.{i}\', block) for i, block in enumerate(base.transformer.h)]\n    roots += [(\'transformer.\' + name, getattr(base.transformer, name))\n              for name in (\'wte\', \'wpe\', \'ln_f\') if hasattr(base.transformer, name)]\n    roots += [(\'projector\', model.prj), (\'lm_head\', head)]\n    with timeline.modules(roots, recursive=False):\n        with timeline.span(\'question_total\'):\n            with timeline.span(\'load_question\', gpu=False):\n                questions = [str(question)]\n            batches = prepare_questions(tokenizer, questions, 1, timeline)\n            result = decode(model, tokenizer, batches, head, mode=mode, device=device,\n                            max_new_tokens=max_new_tokens, latent_iterations=latent_iterations, timeline=timeline)\n    timeline.resolve()\n    sample = partition_question(timeline.records)\n    sample.update(arm=arm, tokens=list(result.token_ids[0]), text=result.texts[0],\n                  prompt_tokens=int(batches[0].mask.sum()),\n                  latent_steps=latent_iterations if mode == \'codi\' else 0,\n                  visible_tokens=result.generated_token_counts[0])\n    return sample, timeline.records\n'
EXPERIMENT_SOURCE_SHA256 = '0521378e5e43bdf2fdac633a49afcb6092b2837a18b665aaa99c2284ab218c51'
runtime_path=pathlib.Path('/kaggle/working/dual_global_head_runtime.py')
runtime_path.write_text(RUNTIME_SOURCE)
spec=importlib.util.spec_from_file_location('dual_global_head_runtime',runtime_path)
runtime=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=runtime
spec.loader.exec_module(runtime)
import torch
import pandas as pd
from dataclasses import asdict
from importlib.metadata import version
from src.mech.global_low_rank_head import (
    NestedLowRankVocabularyHead,activation_whitened_factors,distil_nested_head,evaluate_nested_head)
from src.models.official_codi import (
    build_official_codi_gpt2,download_official_checkpoint,load_official_checkpoint,official_codi_base_model)
from src.inference.official_codi_fast import (
    generate_official_codi_fast,prepare_official_codi_batches,merge_official_codi_lora_)
from src.utils.config import load_config
from src.data.answer_extract import answers_match,normalize_gold
assert torch.cuda.is_available(),'Select Settings > Accelerator > GPU T4 x2.'
assert 'T4' in torch.cuda.get_device_name(0),'Select GPU T4 x2 in Kaggle settings and restart the session.'
device=torch.device('cuda:0')
DEPLOY_DTYPE=torch.float16
torch.manual_seed(SEED); random.seed(SEED)
config=dict(seed=SEED,fit=FIT_QUESTIONS,selection=SELECT_QUESTIONS,recovery=RECOVERY_QUESTIONS,
    state_caps=[MAX_FIT_STATES,MAX_SELECT_STATES,MAX_RECOVERY_STATES],ranks=RANKS,
    epochs=[CLEAN_EPOCHS,RECOVERY_EPOCHS],token_caps=MAX_NEW_TOKENS,
    questions=TIMING_QUESTIONS,repeats=TIMING_REPEATS,batch_size=1,heads=['dense','rank96_eager'],accuracy='full_gsm8k_test',
    base=BASE_COMMIT,source=EXPERIMENT_SOURCE_SHA256,torch=torch.__version__,cuda=torch.version.cuda,
    packages={name:version(name) for name in ('transformers','peft','huggingface_hub','hf_xet','accelerate')},
    gpu=torch.cuda.get_device_name(0))
RUN_DIR=OUTPUT_ROOT/hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:16]
RUN_DIR.mkdir(parents=True,exist_ok=True)
DEBUG_PATH=RUN_DIR/'debug.jsonl.gz'
# Each Run All starts a fresh measurement log; fitted heads in the same run folder are reused.
with gzip.open(DEBUG_PATH,'wt') as stream: pass

def debug_record(kind,payload):
    with gzip.open(DEBUG_PATH,'at',encoding='utf-8') as stream:
        stream.write(json.dumps(dict(kind=kind,data=payload),default=str)+'\n')
def save_json(path,value):
    temp=pathlib.Path(str(path)+'.tmp'); temp.write_text(json.dumps(value,indent=2,default=str)); temp.replace(path)
def save_pt(path,value):
    temp=pathlib.Path(str(path)+'.tmp'); torch.save(value,temp); temp.replace(path)
setup=runtime.Timeline(device)
setup.context.update(mode='setup')
setup.records.extend(BOOTSTRAP_TIMES)
def flush_setup():
    setup.resolve()
    for row in setup.records: debug_record('setup',row)
    setup.records.clear()
debug_record('manifest',config)
debug_record('dependency_log',setup_log.read_text())
flush_setup()
# These two construction notices are expected in the pinned official loader.
# Preserve them in the debug log and keep unrelated warnings visible.
class KnownConstructionNotice(logging.Filter):
    def filter(self,record):
        if record.getMessage().startswith('The new embeddings will be initialized'):
            debug_record('construction_notice',record.getMessage()); return False
        return True
logging.getLogger('transformers.modeling_utils').addFilter(KnownConstructionNotice())
print(f"Using {config['gpu']}; rank 96, FP16, batch 1. Fitting is the slow setup step.")
from src.data.answer_extract import answers_match,normalize_gold

In [ ]:
DATA_REVISION='3101c7d5072418e28b9008a6636bde82a006892c'
url=f'https://raw.githubusercontent.com/openai/grade-school-math/{DATA_REVISION}/grade_school_math/data/train.jsonl'
with setup.span('gsm8k_download_parse_and_partition',gpu=False):
    with urlopen(url,timeout=300) as response:
        train=[json.loads(line) for line in response if line.strip()]
    unique={}
    for row in train:
        key=' '.join(row['question'].casefold().split())
        unique.setdefault(key,dict(question=str(row['question']),gold=str(row['answer'])))
    rows=list(unique.values()); random.Random(SEED).shuffle(rows)
    splits={}; start=0
    for name,size in zip(['fit','selection','recovery','timing','warmup'],
                         [FIT_QUESTIONS,SELECT_QUESTIONS,RECOVERY_QUESTIONS,TIMING_QUESTIONS,4]):
        splits[name]=rows[start:start+size]; start+=size
    assert start<=len(rows)
    debug_record('partitions',splits)
with setup.span('gsm8k_test_download_and_parse',gpu=False):
    with urlopen(url.replace('train.jsonl','test.jsonl'),timeout=300) as response:
        test=[json.loads(line) for line in response if line.strip()]
    test_rows=[dict(question=str(row['question']),gold=str(normalize_gold(row['answer'],'gsm8k_main'))) for row in test]
    assert len(test_rows)==1319 and all(row['gold']!='None' for row in test_rows)
    train_keys={' '.join(r['question'].casefold().split()) for r in train}
    assert not train_keys & {' '.join(r['question'].casefold().split()) for r in test_rows}
    debug_record('accuracy_questions',test_rows)

with setup.span('config_load',gpu=False): cfg=load_config('configs/official_codi_gpt2.yaml')
with setup.span('checkpoint_download_and_hash',gpu=False):
    checkpoint=download_official_checkpoint(repo_id=cfg.checkpoint.repo_id,revision=cfg.checkpoint.revision,
        filename=cfg.checkpoint.filename,expected_sha256=cfg.checkpoint.sha256)
with setup.span('gpt2_model_and_tokenizer_load',gpu=False):
    with warnings.catch_warnings(record=True) as notices:
        warnings.filterwarnings('always',message='fan_in_fan_out is set to False.*')
        model,tokenizer=build_official_codi_gpt2(base_model=cfg.model.base_model,base_revision=cfg.model.base_revision,
            dtype=torch.float32,settings=cfg.model)
    for notice in notices:
        if 'fan_in_fan_out is set to False' in str(notice.message):
            debug_record('construction_notice',str(notice.message))
        else: warnings.warn(str(notice.message),notice.category)
with setup.span('checkpoint_load_and_verification',gpu=False):
    load_report=load_official_checkpoint(model,checkpoint,expected_sha256=cfg.checkpoint.sha256)
    debug_record('checkpoint',asdict(load_report))
with setup.span('model_host_to_device_fp32'): model.requires_grad_(False).to(device).eval()
base=official_codi_base_model(model)
full_head=base.get_output_embeddings()
weight=full_head.weight[:model.eot_id].detach()
bias=None if getattr(full_head,'bias',None) is None else full_head.bias[:model.eot_id].detach()
flush_setup()

In [ ]:
parity_questions=[r['question'] for r in splits['selection'][:4]]
parity={}
with torch.no_grad():
    for mode in MODES:
        prepared=runtime.prepare_questions(tokenizer,parity_questions,4)
        observed=runtime.decode(model,tokenizer,prepared,full_head,mode=mode,device=device,
            max_new_tokens=MAX_NEW_TOKENS[mode],latent_iterations=6)
        if mode=='codi':
            reference=generate_official_codi_fast(model,tokenizer,
                prepare_official_codi_batches(tokenizer,parity_questions,batch_size=4,length_bucketed=False),
                latent_iterations=6,max_new_tokens=MAX_NEW_TOKENS[mode],device=device,answer_cue='The answer is:')
            expected=reference.token_ids
        else:
            from transformers import LogitsProcessor,LogitsProcessorList
            class VocabularyBoundary(LogitsProcessor):
                def __call__(self,input_ids,scores):
                    scores[:,int(model.eot_id):]=float('-inf'); return scores
            batch=prepared[0]
            generated=base.generate(input_ids=batch.ids.to(device),attention_mask=batch.mask.to(device),
                do_sample=False,max_new_tokens=MAX_NEW_TOKENS[mode],pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,logits_processor=LogitsProcessorList([VocabularyBoundary()]))
            expected=[]
            for seq in generated[:,batch.ids.shape[1]:].cpu().tolist():
                if tokenizer.eos_token_id in seq: seq=seq[:seq.index(tokenizer.eos_token_id)+1]
                expected.append(tuple(seq))
            expected=tuple(expected)
        parity[mode]=dict(examples=len(expected),exact=observed.token_ids==expected)
        assert parity[mode]['exact'],f'{mode}: custom decoder differs from reference; stop before fitting'
debug_record('decoder_parity',parity)
print('Decoder checks passed: CODI and explicit GPT-2.')

In [ ]:
def collect(mode,head,population,cap,tag):
    path=RUN_DIR/f'states_{mode}_{tag}.pt'
    if path.exists():
        with setup.span('state_cache_disk_load',gpu=False,mode=mode,tag=tag):
            return torch.load(path,map_location='cpu',weights_only=False)
    chunks=[]; positions=[]
    def observe(hidden,active,position,indices):
        chunks.append(hidden[active].detach().cpu().float())
        positions.extend([position]*int(active.sum()))
    questions=[r['question'] for r in splits[population]]
    batches=runtime.prepare_questions(tokenizer,questions,COLLECT_BATCH_SIZE)
    with setup.span('trajectory_collection',mode=mode,population=population):
        runtime.decode(model,tokenizer,batches,head,mode=mode,device=device,
                       max_new_tokens=MAX_NEW_TOKENS[mode],observer=observe)
    values=torch.cat(chunks)
    indices=torch.randperm(len(values),generator=torch.Generator().manual_seed(SEED))[:cap]
    result=dict(states=values[indices].clone(),positions=torch.tensor(positions)[indices],observed_states=len(values))
    with setup.span('state_cache_disk_save',gpu=False,mode=mode,tag=tag): save_pt(path,result)
    return result

def evaluate_positions(head,bundle):
    result={}
    for rank in RANKS:
        result[str(rank)]={}
        for label,mask in [('all',torch.ones(len(bundle['states']),dtype=torch.bool)),
                           ('p0',bundle['positions']==0),('p1',bundle['positions']==1),('p2plus',bundle['positions']>=2)]:
            result[str(rank)][label]=dict(states=int(mask.sum()),**(evaluate_nested_head(
                head,bundle['states'][mask],weight,readout_bias=bias,rank=rank,batch_size=DISTILL_BATCH_SIZE)
                if mask.any() else {}))
    return result

for mode in MODES:
    artifact=RUN_DIR/f'global_head_{mode}.pt'
    if artifact.exists(): print('Reuse fitted head:',mode); continue
    print('Collect/fitting:',mode,flush=True)
    fit=collect(mode,full_head,'fit',MAX_FIT_STATES,'fit')
    selection=collect(mode,full_head,'selection',MAX_SELECT_STATES,'selection')
    with setup.span('activation_whitened_initialization',mode=mode):
        centre,down,up,out_bias,init=activation_whitened_factors(fit['states'],weight,96,
            readout_bias=bias,seed=SEED,compute_device=device)
        head=NestedLowRankVocabularyHead.from_whitened_factors(centre,down,up,out_bias,RANKS).to(device)
    initial_metrics=evaluate_positions(head,selection)
    with setup.span('clean_distillation',mode=mode):
        clean=distil_nested_head(head,fit['states'],selection['states'],weight,readout_bias=bias,
            epochs=CLEAN_EPOCHS,batch_size=DISTILL_BATCH_SIZE,learning_rate=2e-4,seed=SEED)
    head.disable_adaptive(); head.set_rank(64)
    recovery=collect(mode,head,'recovery',MAX_RECOVERY_STATES,'onpolicy')
    with setup.span('recovery_distillation',mode=mode):
        recovered=distil_nested_head(head,torch.cat((fit['states'],recovery['states'])),
            selection['states'],weight,readout_bias=bias,epochs=RECOVERY_EPOCHS,
            batch_size=DISTILL_BATCH_SIZE,learning_rate=2e-4,seed=SEED+1)
    report=dict(mode=mode,initialization=asdict(init),clean=asdict(clean),recovery=asdict(recovered),
                initial=initial_metrics,final=evaluate_positions(head,selection),
                fit_states=len(fit['states']),recovery_states=len(recovery['states']))
    with setup.span('trained_head_disk_save',gpu=False,mode=mode):
        save_pt(artifact,dict(state_dict={k:v.detach().cpu().clone() for k,v in head.state_dict().items()},report=report))
    flush_setup()
    del head,fit,selection,recovery,centre,down,up,out_bias
    gc.collect(); torch.cuda.empty_cache()
with setup.span('lora_merge'): merge_official_codi_lora_(model)
with setup.span('model_fp16_conversion'): model.to(dtype=DEPLOY_DTYPE).eval()
base=official_codi_base_model(model); full_head=base.get_output_embeddings()
del weight,bias
flush_setup()

Load the optimization engine and the same fitted heads. Compiler/graph setup is excluded from clean timing.

In [ ]:
OPTIMIZER_SOURCE = '"""Static GPT-2 decode, compiler fusion, CUDA graphs and isolated diagnostics.\n\nBatch one only. This preserves CODI\'s six latent steps and explicit teacher prefix.\nMain runs install no module hooks. Lossy weight kernels are separate candidates.\n"""\nfrom contextlib import contextmanager, nullcontext\nimport math\nimport time\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\ntry:\n    import dual_global_head_runtime as reference\nexcept ModuleNotFoundError:\n    from src.inference import global_head_comparison as reference\nfrom src.models.official_codi import official_codi_base_model\ntry:\n    import triton\n    import triton.language as tl\nexcept ImportError:\n    triton = None\n\nif triton is not None:\n    @triton.jit\n    def _quant_gemv(X, W, SCALE, BIAS, Y, K:tl.constexpr, N:tl.constexpr,\n                    BITS:tl.constexpr, BK:tl.constexpr, BN:tl.constexpr):\n        rows=tl.program_id(0)*BN+tl.arange(0,BN)\n        cols=tl.arange(0,BK)\n        x=tl.load(X+cols,cols<K,0).to(tl.float32)\n        if BITS == 4:\n            packed=tl.load(W+rows[:,None]*(K//2)+cols[None,:]//2,\n                           (rows[:,None]<N)&(cols[None,:]<K),0)\n            w=((packed >> ((cols[None,:]%2)*4)) & 15).to(tl.float32)-8.0\n        else:\n            w=tl.load(W+rows[:,None]*K+cols[None,:],\n                      (rows[:,None]<N)&(cols[None,:]<K),0).to(tl.float32)\n        scale=tl.load(SCALE+rows,rows<N,0)\n        bias=tl.load(BIAS+rows,rows<N,0).to(tl.float32)\n        y=tl.sum(w*x[None,:],axis=1)*scale+bias\n        tl.store(Y+rows,y,rows<N)\n\n\nclass PackedLinear(nn.Module):\n    """Contiguous Linear layout; optional per-output INT8/INT4 decode GEMV.\n\n    Prefill uses the SAME quantized values, dequantized once at construction.\n    Keeping that prefill copy means this is a bandwidth experiment, not a claim\n    that total model VRAM shrinks by 2x/4x.\n    """\n    def __init__(self, source, bits=16):\n        super().__init__()\n        # GPT-2 Conv1D stores [in, out], torch Linear stores [out, in].\n        weight=source.weight.detach()\n        weight=weight if isinstance(source,nn.Linear) else weight.t()\n        weight=weight.contiguous().clone()\n        bias=source.bias.detach().clone() if source.bias is not None else weight.new_zeros(weight.shape[0])\n        self.bits=bits\n        if bits not in (4,8,16): raise ValueError(bits)\n        if bits<16:\n            bound=7 if bits==4 else 127\n            scale=weight.float().abs().amax(1).clamp_min(1e-8)/bound\n            q=(weight.float()/scale[:,None]).round().clamp(-bound,bound).to(torch.int8)\n            effective=(q.float()*scale[:,None]).to(weight.dtype)\n            if bits==4:\n                if weight.shape[1]%2: raise ValueError(\'INT4 requires even input width\')\n                shifted=(q.to(torch.int16)+8).to(torch.uint8)\n                packed=shifted[:,0::2] | (shifted[:,1::2] << 4)\n            else: packed=q\n            self.register_buffer(\'packed\',packed.contiguous())\n            self.register_buffer(\'scale\',scale.contiguous())\n            weight=effective\n        self.register_buffer(\'weight\',weight)\n        self.register_buffer(\'bias\',bias)\n\n    def forward(self,x):\n        if self.bits<16 and x.is_cuda and x.numel()==self.weight.shape[1]:\n            if triton is None: raise RuntimeError(\'Triton is required for packed GPU weights\')\n            n,k=self.weight.shape\n            output=torch.empty((*x.shape[:-1],n),device=x.device,dtype=x.dtype)\n            _quant_gemv[(triton.cdiv(n,4),)](x,self.packed,self.scale,self.bias,output,\n                k,n,self.bits,triton.next_power_of_2(k),4,num_warps=4)\n            return output\n        return F.linear(x,self.weight,self.bias)\n\n\nclass StaticBlock(nn.Module):\n    def __init__(self, source, bits=16):\n        super().__init__()\n        self.ln_1=source.ln_1; self.ln_2=source.ln_2\n        self.qkv=PackedLinear(source.attn.c_attn,bits)\n        self.attn_out=PackedLinear(source.attn.c_proj,bits)\n        self.fc=PackedLinear(source.mlp.c_fc,bits)\n        self.proj=PackedLinear(source.mlp.c_proj,bits)\n        self.act=source.mlp.act\n        self.heads=source.attn.num_heads\n        self.width=source.attn.head_dim\n        self.scale=(self.width**-0.5 if source.attn.scale_attn_weights else 1.0)\n        if source.attn.scale_attn_by_inverse_layer_idx:\n            self.scale/=source.attn.layer_idx+1\n        if source.attn.reorder_and_upcast_attn:\n            raise ValueError(\'This engine requires the released GPT-2 attention settings\')\n\n    def forward(self,x,k_cache,v_cache,positions,mask=None):\n        normalized=self.ln_1(x)\n        q,k,v=self.qkv(normalized).chunk(3,-1)\n        shape=(1,x.shape[1],self.heads,self.width)\n        q,k,v=(t.view(shape).transpose(1,2) for t in (q,k,v))\n        k_cache.index_copy_(2,positions,k)\n        v_cache.index_copy_(2,positions,v)\n        # The mask is explicit for a static cache; is_causal=True is incorrect\n        # for a one-token query attending to a longer preallocated key tensor.\n        attended=F.scaled_dot_product_attention(q,k_cache,v_cache,attn_mask=mask,\n                    dropout_p=0.,is_causal=False,scale=self.scale)\n        attended=attended.transpose(1,2).contiguous().view(1,x.shape[1],-1)\n        x=x+self.attn_out(attended)\n        return x+self.proj(self.act(self.fc(self.ln_2(x))))\n\n\nclass StaticCore(nn.Module):\n    def __init__(self, model, bits=16):\n        super().__init__()\n        base=official_codi_base_model(model)\n        body=base.transformer\n        self.wte=body.wte; self.wpe=body.wpe; self.ln_f=body.ln_f\n        self.blocks=nn.ModuleList([StaticBlock(b,bits) for b in body.h])\n        self.capacity=int(model.config.n_positions)\n        self.hidden_size=int(model.config.hidden_size)\n        device,dtype=body.wte.weight.device,body.wte.weight.dtype\n        shape=(len(self.blocks),1,model.config.n_head,self.capacity,self.hidden_size//model.config.n_head)\n        self.register_buffer(\'keys\',torch.zeros(shape,device=device,dtype=dtype))\n        self.register_buffer(\'values\',torch.zeros_like(self.keys))\n        self.register_buffer(\'slots\',torch.arange(self.capacity,device=device))\n        self.register_buffer(\'decode_mask\',torch.empty((1,1,1,self.capacity),device=device,dtype=torch.bool))\n        self.requires_grad_(False).eval()\n\n    def forward(self,embedded,position):\n        positions=position.clamp_max(self.capacity-1)\n        if embedded.shape[1]==1:\n            torch.le(self.slots.reshape(1,1,1,-1),positions.reshape(1,1,1,1),out=self.decode_mask)\n            mask=self.decode_mask\n        else:\n            mask=(self.slots<=positions.reshape(-1,1)).reshape(1,1,embedded.shape[1],self.capacity)\n        x=embedded+self.wpe(positions)\n        for i,block in enumerate(self.blocks):\n            x=block(x,self.keys[i],self.values[i],positions,mask)\n        return self.ln_f(x)\n\n    def prefill(self,ids):\n        self.keys.zero_(); self.values.zero_()\n        positions=torch.arange(ids.shape[1],device=ids.device)\n        return self(self.wte(ids),positions)\n\n\nclass GraphSequence:\n    """One GPU replay for a sequence; optional events only in diagnostic graphs."""\n    def __init__(self,operations,reset,diagnostic=False):\n        self.operations=operations; self.pairs=[]\n        device=torch.cuda.current_device()\n        stream=torch.cuda.Stream(device=device)\n        stream.wait_stream(torch.cuda.current_stream(device))\n        with torch.cuda.stream(stream):\n            for _ in range(3):\n                reset()\n                for _,fn in operations: fn()\n        torch.cuda.current_stream(device).wait_stream(stream)\n        torch.cuda.synchronize(device)\n        if diagnostic:\n            # External events must become record nodes, rather than only internal\n            # capture dependencies, to measure successive graph executions.\n            self.pairs=[(name,torch.cuda.Event(enable_timing=True,external=True),\n                         torch.cuda.Event(enable_timing=True,external=True)) for name,_ in operations]\n        reset()\n        self.graph=torch.cuda.CUDAGraph()\n        with torch.cuda.graph(self.graph):\n            for i,(_,fn) in enumerate(operations):\n                if diagnostic: self.pairs[i][1].record()\n                fn()\n                if diagnostic: self.pairs[i][2].record()\n        torch.cuda.synchronize(device)\n\n    def __call__(self):\n        self.graph.replay()\n\n    def durations(self):\n        torch.cuda.synchronize()\n        return [(name,start.elapsed_time(end)) for name,start,end in self.pairs]\n\n\nclass OptimizedDecoder:\n    def __init__(self,model,tokenizer,head,*,compile_steps=False,cuda_graphs=False,\n                 bits=16,chunk_size=1,compiler_backend=\'inductor\'):\n        self.model=model; self.tokenizer=tokenizer; self.head=head\n        self.core=StaticCore(model,bits)\n        self.device=self.core.keys.device\n        self.dtype=self.core.keys.dtype\n        self.chunk_size=chunk_size\n        self.compile_steps=compile_steps; self.cuda_graphs=cuda_graphs\n        self.bits=bits; self.graphs={}; self.diagnostic_graphs={}\n        self.diagnostic_graph_error=None\n        d=self.core.hidden_size\n        self.current=torch.zeros((1,1,d),device=self.device,dtype=self.dtype)\n        self.hidden=torch.zeros_like(self.current)\n        self.latent=torch.zeros_like(self.current)\n        self.position=torch.zeros(1,device=self.device,dtype=torch.long)\n        self.chosen=torch.zeros(1,device=self.device,dtype=torch.long)\n        self.cursor=torch.zeros(1,device=self.device,dtype=torch.long)\n        self.count=torch.zeros_like(self.cursor)\n        self.limit=torch.ones_like(self.cursor)\n        self.finished=torch.zeros(1,device=self.device,dtype=torch.bool)\n        self.output=torch.empty(self.core.capacity+chunk_size,device=self.device,dtype=torch.long)\n        self.cue_ids=tokenizer(\' The answer is:\',add_special_tokens=False)[\'input_ids\']\n        self.cue=torch.tensor([model.eot_id,*self.cue_ids],device=self.device).reshape(1,-1)\n        self.eos=int(tokenizer.eos_token_id)\n        self.functions={\'body\':self._body,\'projector\':self._projector,\'head\':self._head,\n                        \'record\':self._record,\'embed\':self._embed,\'latent_input\':self._latent_input,\'cue\':self._cue}\n        if compile_steps:\n            # Manual graph capture below includes mutable KV/input buffers. Do not\n            # nest automatic cudagraphs from reduce-overhead inside our graphs.\n            for name in (\'body\',\'projector\',\'head\',\'record\',\'embed\',\'cue\'):\n                kwargs=dict(fullgraph=True,dynamic=False,backend=compiler_backend)\n                if compiler_backend==\'inductor\': kwargs[\'options\']={\'triton.cudagraphs\':False}\n                self.functions[name]=torch.compile(self.functions[name],**kwargs)\n        if cuda_graphs and self.device.type!=\'cuda\': raise ValueError(\'CUDA graphs need a CUDA device\')\n\n    def _body(self):\n        self.hidden.copy_(self.core(self.current,self.position))\n        self.position.add_(1)\n    def _cue(self):\n        positions=self.position+torch.arange(self.cue.shape[1],device=self.device)\n        hidden=self.core(self.core.wte(self.cue),positions)\n        self.hidden.copy_(hidden[:,-1:,:])\n        self.position.add_(self.cue.shape[1])\n    def _projector(self):\n        self.latent.copy_(self.model.prj(self.hidden))\n    def _latent_input(self):\n        self.current.copy_(self.latent)\n    def _head(self):\n        logits=self.head(self.hidden[:,-1,:])\n        self.chosen.copy_(logits[...,:int(self.model.eot_id)].argmax(-1))\n    def _embed(self):\n        self.current.copy_(self.core.wte(self.chosen).unsqueeze(1))\n    def _record(self):\n        active=(~self.finished)&(self.cursor<self.limit)\n        token=torch.where(active,self.chosen,self.eos)\n        self.output.index_copy_(0,self.cursor.clamp_max(self.output.numel()-1),token)\n        self.count.add_(active.long())\n        self.cursor.add_(1)\n        self.finished.logical_or_((token==self.eos)|(self.cursor>=self.limit))\n\n    def operations(self,kind):\n        f=self.functions\n        if kind==\'latent\':\n            operations=[(\'Latent projector\',f[\'projector\'])]\n            for _ in range(6):\n                operations += [(\'Buffers / embeddings\',f[\'latent_input\']),(\'Transformer\',f[\'body\']),(\'Latent projector\',f[\'projector\'])]\n            return operations\n        if kind==\'cue\': return [(\'Transformer\',f[\'cue\'])]\n        one=[(\'LM head + argmax\',f[\'head\']),(\'Buffers / embeddings\',f[\'record\']),(\'Buffers / embeddings\',f[\'embed\'])]\n        if kind==\'first\': return one\n        if kind==\'decode\': return ([(\'Transformer\',f[\'body\'])]+one)*self.chunk_size\n        raise ValueError(kind)\n\n    def reset(self):\n        self.position.zero_(); self.cursor.zero_(); self.count.zero_(); self.finished.zero_()\n        self.limit.fill_(self.core.capacity)\n        self.current.zero_(); self.hidden.zero_(); self.latent.zero_()\n\n    @torch.inference_mode()\n    def prepare(self):\n        for kind in (\'latent\',\'cue\',\'first\',\'decode\'):\n            self.reset()\n            for _,fn in self.operations(kind): fn()\n            if self.cuda_graphs:\n                self.graphs[kind]=GraphSequence(self.operations(kind),self.reset)\n        self.reset()\n\n    @torch.inference_mode()\n    def prepare_diagnostics(self):\n        if not self.cuda_graphs: return\n        try:\n            for kind in (\'latent\',\'cue\',\'first\',\'decode\'):\n                self.diagnostic_graphs[kind]=GraphSequence(self.operations(kind),self.reset,diagnostic=True)\n        except (TypeError,RuntimeError) as error:\n            self.diagnostic_graphs.clear()\n            self.diagnostic_graph_error=str(error)\n        self.reset()\n\n    def execute(self,kind,diagnostics=None,phase=\'visible\'):\n        if self.cuda_graphs and diagnostics is None:\n            self.graphs[kind]()\n        elif diagnostics is None:\n            for _,fn in self.operations(kind): fn()\n        elif self.cuda_graphs and self.diagnostic_graphs:\n            started=time.perf_counter()\n            self.diagnostic_graphs[kind]()\n            durations=self.diagnostic_graphs[kind].durations()\n            wall=1000*(time.perf_counter()-started)\n            for name,elapsed in durations:\n                diagnostics.append(dict(name=name,phase=phase,ms=elapsed,gpu_ms=elapsed))\n            diagnostics.append(dict(name=\'Runtime / measurement overhead\',phase=phase,\n                                    ms=wall-sum(value for _,value in durations)))\n        else:\n            for name,fn in self.operations(kind):\n                with measured(diagnostics,name,phase,self.device): fn()\n\n    @torch.inference_mode()\n    def generate(self,question,mode,max_new_tokens,diagnostic=False):\n        if mode not in (\'codi\',\'explicit_cot\'): raise ValueError(mode)\n        parts=[] if diagnostic else None\n        with measured(parts,\'Preparation / transfers\',\'shared\',self.device):\n            ids=reference.prepare_questions(self.tokenizer,[question],1)[0].ids.to(self.device)\n            if mode==\'codi\': ids=torch.cat((ids,ids.new_full((1,1),int(self.model.bot_id))),1)\n            reserve=6+self.cue.shape[1] if mode==\'codi\' else 0\n            if max_new_tokens<1 or ids.shape[1]+reserve+max_new_tokens>self.core.capacity:\n                raise ValueError(\'Prompt + generation exceeds GPT-2 context\')\n            self.cursor.zero_(); self.count.zero_(); self.finished.zero_(); self.limit.fill_(max_new_tokens)\n        with measured(parts,\'Transformer\',\'shared\',self.device):\n            self.hidden.copy_(self.core.prefill(ids)[:,-1:,:])\n            self.position.fill_(ids.shape[1])\n        if mode==\'codi\':\n            self.execute(\'latent\',parts,\'latent\')\n            self.execute(\'cue\',parts,\'visible\')\n        self.execute(\'first\',parts,\'visible\')\n        # One host EOS check per chunk, outside the captured graph. CODI uses 1\n        # by default; explicit may use 4. Extra work after EOS is masked/countless.\n        while not bool(self.finished.item()):\n            self.execute(\'decode\',parts,\'visible\')\n        with measured(parts,\'Output / host work\',\'visible\',self.device):\n            count=int(self.count.item())\n            tokens=tuple(self.output[:count].cpu().tolist())\n            text=self.tokenizer.decode(tokens,skip_special_tokens=True)\n        return dict(tokens=tokens,text=text,visible_tokens=count,latent_steps=6 if mode==\'codi\' else 0,\n                    parts=parts or [],mode=mode)\n\n\n@contextmanager\ndef measured(records,name,phase,device):\n    if records is None:\n        yield; return\n    if torch.device(device).type==\'cuda\':\n        start,end=torch.cuda.Event(enable_timing=True),torch.cuda.Event(enable_timing=True)\n        wall=time.perf_counter()\n        start.record()\n        try: yield\n        finally:\n            end.record(); end.synchronize()\n            records.append(dict(name=name,phase=phase,ms=1000*(time.perf_counter()-wall),gpu_ms=start.elapsed_time(end)))\n    else:\n        start=time.perf_counter()\n        try: yield\n        finally: records.append(dict(name=name,phase=phase,ms=(time.perf_counter()-start)*1000))\n\n\ndef synchronize(device):\n    if torch.device(device).type==\'cuda\': torch.cuda.synchronize(device)\n\n\n@torch.inference_mode()\ndef baseline_generate(model,tokenizer,head,question,mode,max_new_tokens,diagnostic=False):\n    device=next(model.parameters()).device\n    trace=reference.Timeline(device,enabled=diagnostic,unified_clock=True)\n    trace.context.update(mode=mode,phase=\'shared\')\n    with trace.span(\'question_total\'):\n        batches=reference.prepare_questions(tokenizer,[question],1,trace)\n        result=reference.decode(model,tokenizer,batches,head,mode=mode,device=device,\n                                max_new_tokens=max_new_tokens,timeline=trace)\n    parts=[]\n    if diagnostic:\n        trace.resolve()\n        rows={r[\'event_id\']:r for r in trace.records}\n        children={}\n        for row in trace.records: children.setdefault(row[\'parent_id\'],[]).append(row)\n        clock=\'cuda_stream_ms\' if device.type==\'cuda\' else \'cpu_wall_ms\'\n        for row in trace.records:\n            value=row[clock]-sum(r[clock] for r in children.get(row[\'event_id\'],[]))\n            name=row[\'name\']; phase=row.get(\'phase\',\'shared\')\n            phase=\'latent\' if phase in (\'latent\',\'latent_projection_initial\') else (\'visible\' if phase in (\'answer_cue\',\'visible_decode\') else \'shared\')\n            if name.startswith(\'transformer_\'): category=\'Transformer\'\n            elif name==\'latent_projector\': category=\'Latent projector\'\n            elif name==\'lm_head_and_argmax\': category=\'LM head + argmax\'\n            elif name.startswith((\'question_\',\'host_to_device\',\'cpu_padding\')): category=\'Preparation / transfers\'\n            elif name.startswith((\'device_to_host\',\'cpu_token\')): category=\'Output / host work\'\n            elif name in (\'question_total\',\'phase_latent\',\'phase_visible\'): category=\'Runtime / measurement overhead\'\n            else: category=\'Buffers / embeddings\'\n            parts.append(dict(name=category,phase=phase,ms=value))\n    return dict(tokens=result.token_ids[0],text=result.texts[0],visible_tokens=result.generated_token_counts[0],\n                latent_steps=6 if mode==\'codi\' else 0,parts=parts,mode=mode)\n\n\ndef timed(call,device):\n    synchronize(device); started=time.perf_counter()\n    result=call()\n    synchronize(device)\n    result[\'wall_ms\']=(time.perf_counter()-started)*1000\n    return result\n\n\nCOARSE_ROWS=(\'Preparation / transfers\',\'Transformer\',\'Latent projector\',\'LM head + argmax\',\n             \'Buffers / embeddings\',\'Output / host work\',\'Runtime / measurement overhead\')\n\n\ndef coarse_table(samples,per_token=False):\n    columns=reference.COLUMNS\n    rows={name:dict.fromkeys(columns,0.) for name in COARSE_ROWS}\n    for mode in (\'explicit_cot\',\'codi\'):\n        group=[s for s in samples if s[\'mode\']==mode]\n        if not group: raise ValueError(\'Both modes required\')\n        visible=sum(s[\'visible_tokens\'] for s in group)\n        latent=sum(s[\'latent_steps\'] for s in group)\n        overall=\'Explicit\' if mode==\'explicit_cot\' else \'CODI overall\'\n        den={overall:(visible+latent if mode==\'codi\' else visible) if per_token else len(group),\n             \'CODI latent only\':latent if per_token else len(group),\n             \'CODI visible only\':visible if per_token else len(group)}\n        for sample in group:\n            measured_sum=sum(p[\'ms\'] for p in sample[\'parts\'])\n            remainder=sample[\'wall_ms\']-measured_sum\n            if remainder < -0.05: raise ValueError(\'Diagnostic spans overlap; cannot present additive totals\')\n            parts=sample[\'parts\']+[dict(name=\'Runtime / measurement overhead\',phase=\'shared\',ms=remainder)]\n            for part in parts:\n                rows[part[\'name\']][overall]+=part[\'ms\']/den[overall]\n                if mode==\'codi\' and part[\'phase\'] in (\'latent\',\'visible\'):\n                    col=\'CODI latent only\' if part[\'phase\']==\'latent\' else \'CODI visible only\'\n                    rows[part[\'name\']][col]+=part[\'ms\']/den[col]\n    totals={col:sum(row[col] for row in rows.values()) for col in columns}\n    return [(\'Total average time\',totals),*rows.items(),(\'Total average time (repeat)\',dict(totals))]\n\n\n@torch.inference_mode()\ndef numerical_gate(model,tokenizer,engine,questions,rtol=0.03,atol=0.03):\n    """Teacher-forced FP16 state checks before trusting free-generation accuracy."""\n    from transformers import DynamicCache\n    base=official_codi_base_model(model)\n    comparisons=0; worst=0.\n    for mode in (\'explicit_cot\',\'codi\'):\n        for question in questions:\n            ids=reference.prepare_questions(tokenizer,[question],1)[0].ids.to(engine.device)\n            if mode==\'codi\': ids=torch.cat((ids,ids.new_full((1,1),int(model.bot_id))),1)\n            cache=DynamicCache()\n            expected=base.transformer(input_ids=ids,past_key_values=cache,use_cache=True).last_hidden_state[:,-1:,:]\n            actual=engine.core.prefill(ids)[:,-1:,:].clone()\n            pos=ids.shape[1]\n            engine.position.fill_(pos)\n            for step in range(8):\n                torch.testing.assert_close(actual,expected,rtol=rtol,atol=atol)\n                worst=max(worst,float((actual-expected).abs().max())); comparisons+=1\n                if mode==\'codi\' and step<6:\n                    a,b=model.prj(actual),model.prj(expected)\n                else:\n                    # Identical forced visible tokens isolate numeric error from divergent policy.\n                    token=base.lm_head(expected[:,-1,:])[...,:int(model.eot_id)].argmax(-1)\n                    a=b=engine.core.wte(token).unsqueeze(1)\n                engine.current.copy_(a)\n                engine.functions[\'body\']()\n                actual=engine.hidden.clone()\n                expected=base.transformer(inputs_embeds=b,past_key_values=cache,use_cache=True).last_hidden_state\n                pos+=1\n    return dict(comparisons=comparisons,max_absolute_error=worst)\n\n\n@torch.inference_mode()\ndef kernel_gate(core):\n    """Exercise each packed CUDA GEMV shape before compiling/graphing it."""\n    results=[]\n    for block in core.blocks[:1]:\n        for name in (\'qkv\',\'attn_out\',\'fc\',\'proj\'):\n            module=getattr(block,name)\n            x=torch.randn((1,1,module.weight.shape[1]),device=module.weight.device,dtype=module.weight.dtype)\n            actual=module(x); expected=F.linear(x,module.weight,module.bias)\n            torch.testing.assert_close(actual,expected,atol=.025,rtol=.025)\n            results.append(dict(name=name,max_absolute_error=float((actual-expected).abs().max())))\n    return results\n\n\n@torch.inference_mode()\ndef operation_microbenchmark(device,dtype,width=768,repeats=3,iterations=200):\n    """Equal weights/MACs, changed matrix shape; not a pure launch-overhead proof."""\n    x=torch.randn((1,width),device=device,dtype=dtype)\n    w=torch.randn((width,12*width),device=device,dtype=dtype)\n    pieces=[t.contiguous() for t in w.split(width,dim=1)]\n    big_out=torch.empty((1,12*width),device=device,dtype=dtype)\n    small_out=[torch.empty((1,width),device=device,dtype=dtype) for _ in range(12)]\n    def big(): torch.mm(x,w,out=big_out)\n    def small():\n        for a,b in zip(pieces,small_out): torch.mm(x,a,out=b)\n    big(); small()\n    torch.testing.assert_close(big_out,torch.cat(small_out,-1),atol=.05,rtol=.03)\n    operations={\'One large matmul\':big,\'Twelve small matmuls\':small}\n    retained=[]\n    if torch.device(device).type==\'cuda\':\n        for name,fn in list(operations.items()):\n            graph=GraphSequence([(name,fn)],lambda:None)\n            retained.append(graph); operations[name+\' + CUDA Graph\']=graph\n    samples=[]\n    for repeat in range(repeats):\n        for name,fn in (list(operations.items()) if repeat%2==0 else reversed(list(operations.items()))):\n            for _ in range(30): fn()\n            synchronize(device)\n            if torch.device(device).type==\'cuda\':\n                start,end=torch.cuda.Event(enable_timing=True),torch.cuda.Event(enable_timing=True)\n                start.record()\n                for _ in range(iterations): fn()\n                end.record(); end.synchronize(); ms=start.elapsed_time(end)/iterations\n            else:\n                start=time.perf_counter()\n                for _ in range(iterations): fn()\n                ms=1000*(time.perf_counter()-start)/iterations\n            samples.append(dict(name=name,repeat=repeat,microseconds=ms*1000))\n    return samples\n\n\n@torch.inference_mode()\ndef block_substeps(model,tokenizer,question,repeats=12):\n    """The ONLY hook-based probe: one original block, one cached position.\n\n    Every handle is removed before returning. Parent durations are made exclusive;\n    attention includes its fused SDPA, cache operations and layout changes.\n    """\n    from transformers import DynamicCache\n    base=official_codi_base_model(model); block=base.transformer.h[0]\n    device=base.transformer.wte.weight.device\n    ids=reference.prepare_questions(tokenizer,[question],1)[0].ids.to(device)\n    cache=DynamicCache()\n    out=base.transformer(input_ids=ids,past_key_values=cache,use_cache=True)\n    token=base.lm_head(out.last_hidden_state[:,-1,:])[...,:int(model.eot_id)].argmax(-1)\n    position=torch.tensor([ids.shape[1]],device=device)\n    x=base.transformer.wte(token).unsqueeze(1)+base.transformer.wpe(position)\n    roots=[(\'Block\',block),(\'LayerNorm 1\',block.ln_1),(\'Attention/cache/reshape\',block.attn),\n           (\'QKV projection\',block.attn.c_attn),(\'Attention output projection\',block.attn.c_proj),\n           (\'LayerNorm 2\',block.ln_2),(\'MLP dispatch\',block.mlp),(\'FC1\',block.mlp.c_fc),\n           (\'GELU\',block.mlp.act),(\'FC2\',block.mlp.c_proj)]\n    for _ in range(5):\n        cache.crop(ids.shape[1]); block(x,past_key_value=cache,cache_position=position,use_cache=True)\n    trace=reference.Timeline(device,unified_clock=True)\n    with trace.modules(roots,recursive=False):\n        for repeat in range(repeats):\n            cache.crop(ids.shape[1])\n            with trace.metadata(repeat=repeat):\n                block(x,past_key_value=cache,cache_position=position,use_cache=True)\n    trace.resolve()\n    assert not any(module._forward_hooks for _,module in roots)\n    children={}\n    for row in trace.records: children.setdefault(row[\'parent_id\'],[]).append(row)\n    clock=\'cuda_stream_ms\' if device.type==\'cuda\' else \'cpu_wall_ms\'\n    rows=[]\n    for row in trace.records:\n        value=row[clock]-sum(c[clock] for c in children.get(row[\'event_id\'],[]))\n        name=\'Residuals / block dispatch\' if row[\'name\']==\'Block\' else row[\'name\']\n        rows.append(dict(name=name,repeat=row[\'repeat\'],microseconds=1000*value))\n    return rows,trace.records\n'
optimizer_path=pathlib.Path('/kaggle/working/transformer_optimization_runtime.py')
optimizer_path.write_text(OPTIMIZER_SOURCE)
spec=importlib.util.spec_from_file_location('transformer_optimization_runtime',optimizer_path)
opt=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=opt
spec.loader.exec_module(opt)
heads={}
for mode in MODES:
    payload=torch.load(RUN_DIR/f'global_head_{mode}.pt',map_location='cpu',weights_only=False)
    nested=NestedLowRankVocabularyHead(int(model.config.hidden_size),int(model.eot_id),RANKS)
    nested.load_state_dict(payload['state_dict'])
    heads[mode]=runtime.FixedRankHead(nested,96).to(device=device,dtype=DEPLOY_DTYPE).eval()
    debug_record('fit_report',payload['report'])
del payload,nested
dense=runtime.DenseSelector(full_head,int(model.eot_id))
VALIDATION_QUESTIONS=64
DIAGNOSTIC_QUESTIONS=2
validation=splits['selection'][:VALIDATION_QUESTIONS]
warmup=[r['question'] for r in splits['warmup']]
print('Core candidates: static KV, compiled fusion, CUDA Graphs, chunked explicit decoding, packed INT8/INT4.')
print('FlashAttention-2 and original Marlin require newer GPUs; this uses PyTorch SDPA and T4-compatible custom GEMV kernels.')

The equal-MAC experiment changes matrix shape and invocation count, while retaining
identical weights and input. It tests whether fragmentation matters; it does not isolate
launch overhead from occupancy/bandwidth. CUDA Graph controls help distinguish them.

In [ ]:
micro=opt.operation_microbenchmark(device,DEPLOY_DTYPE)
debug_record('equal_mac_samples',micro)
print('Equal arithmetic: one 768x9216 projection versus twelve 768x768 projections.')
display(pd.DataFrame(micro).groupby('name',sort=False)['microseconds'].mean().to_frame('Mean microseconds'))

**Isolated block probe:** one cached position in block 1, on a real question prefix.
These diagnostic component times include probe overhead. Attention groups SDPA,
softmax, cache updates and reshapes; a fused kernel cannot be meaningfully split into
separate QK/softmax/AV wall times. These hooks never enter full-model benchmark runs.

In [ ]:
substeps,substep_events=opt.block_substeps(model,tokenizer,warmup[0])
debug_record('block_substeps',substep_events)
print('Original transformer block 1 - isolated diagnostic, microseconds per call:')
block_table=pd.DataFrame(substeps).groupby('name',sort=False)['microseconds'].mean().to_frame('Mean microseconds')
block_table.loc['Total block (includes probe overhead)']=block_table.sum()
display(block_table)
assert not any(m._forward_hooks or m._forward_pre_hooks for m in model.modules()),'Probe hooks leaked'

Choose the fastest passing candidate per reasoning mode using training-split
validation and separate warmup questions. Test answers and timing questions never
select a candidate. FP16 candidates must pass state checks and at least 95% exact
sequence agreement; all candidates must match or exceed baseline validation accuracy.
Quantization changes weights, so its accuracy is measured explicitly. Failures are
shown, with the full reason retained in the single debug log.

In [ ]:
def baseline(mode,arm,question,diagnostic=False):
    head=dense if arm=='dense' else heads[mode]
    return opt.baseline_generate(model,tokenizer,head,question,mode,MAX_NEW_TOKENS[mode],diagnostic)

def correctness(result,row):
    gold=normalize_gold(row['gold'],'gsm8k_main')
    return answers_match(result['text'],gold)

selected={}; candidate_rows=[]; selection_baselines={}
for mode in MODES:
    expected=[baseline(mode,'rank96',r['question']) for r in validation]
    baseline_correct=sum(correctness(result,row) for result,row in zip(expected,validation))
    for question in warmup: baseline(mode,'rank96',question)
    baseline_ms=sum(opt.timed(lambda q=q:baseline(mode,'rank96',q),device)['wall_ms'] for _ in range(2) for q in warmup)/(2*len(warmup))
    selection_baselines[mode]=dict(correct=baseline_correct,ms=baseline_ms)
    best=None; best_dense=None; best_ms=baseline_ms; best_config=None
    configurations=[('static_cache',dict()),('compiled',dict(compile_steps=True)),
                    ('cuda_graphs',dict(compile_steps=True,cuda_graphs=True))]
    if mode=='explicit_cot': configurations.append(('cuda_graphs_chunk4',dict(compile_steps=True,cuda_graphs=True,chunk_size=4)))
    for bits in (8,4):
        configurations.append((f'packed_int{bits}',dict(compile_steps=True,cuda_graphs=True,bits=bits,chunk_size=4 if mode=='explicit_cot' else 1)))
    candidate_rows.append(dict(mode=mode,candidate='current',status='reference',validation_correct=baseline_correct,
                               validation_total=len(validation),mean_ms=baseline_ms,sequence_agreement=1.))
    for name,settings in configurations:
        engine=None; plain=None; dense_trial=None; plain_dense=None
        print(f'Preparing {mode}/{name} ...',flush=True)
        try:
            if settings.get('bits',16)<16 and opt.triton is None: raise RuntimeError('Triton unavailable for packed GEMV')
            started=time.perf_counter()
            engine=opt.OptimizedDecoder(model,tokenizer,heads[mode],**settings)
            if settings.get('bits',16)<16: debug_record('packed_kernel_check',dict(mode=mode,candidate=name,checks=opt.kernel_gate(engine.core)))
            engine.prepare()
            if settings.get('bits',16)==16:
                debug_record('state_parity',dict(mode=mode,candidate=name,**opt.numerical_gate(model,tokenizer,engine,warmup[:2])))
            # Every candidate is compared against its uncaptured implementation as
            # an additional graph/cache check, including lossy candidates.
            plain=opt.OptimizedDecoder(model,tokenizer,heads[mode],bits=settings.get('bits',16),chunk_size=settings.get('chunk_size',1))
            for question in warmup[:2]:
                observed=engine.generate(question,mode,MAX_NEW_TOKENS[mode])
                control=plain.generate(question,mode,MAX_NEW_TOKENS[mode])
                if observed['tokens']!=control['tokens']: raise RuntimeError('Compiled/graph output differs from the same eager candidate')
            del plain
            results=[engine.generate(row['question'],mode,MAX_NEW_TOKENS[mode]) for row in validation]
            correct=sum(correctness(result,row) for result,row in zip(results,validation))
            agreement=sum(a['tokens']==b['tokens'] for a,b in zip(results,expected))/len(expected)
            for question in warmup: engine.generate(question,mode,MAX_NEW_TOKENS[mode])
            values=[opt.timed(lambda q=q:engine.generate(q,mode,MAX_NEW_TOKENS[mode]),device)['wall_ms'] for _ in range(2) for q in warmup]
            elapsed=sum(values)/len(values)
            passes=correct>=baseline_correct and (settings.get('bits',16)<16 or agreement>=.95)
            status='passed' if passes else 'rejected: accuracy/agreement'
            # Confirm the same transformer configuration also runs with the dense
            # control BEFORE selecting it; do not silently mix execution backends.
            if passes and elapsed<best_ms and elapsed<.98*baseline_ms:
                dense_trial=opt.OptimizedDecoder(model,tokenizer,dense,**settings)
                dense_trial.prepare()
                plain_dense=opt.OptimizedDecoder(model,tokenizer,dense,bits=settings.get('bits',16),chunk_size=settings.get('chunk_size',1))
                for question in warmup[:2]:
                    if dense_trial.generate(question,mode,MAX_NEW_TOKENS[mode])['tokens']!=plain_dense.generate(question,mode,MAX_NEW_TOKENS[mode])['tokens']:
                        raise RuntimeError('Dense control differs from its eager candidate')
                best=engine; best_dense=dense_trial; best_ms=elapsed; best_config=settings; best_name=name
            candidate_rows.append(dict(mode=mode,candidate=name,status=status,validation_correct=correct,
                validation_total=len(validation),mean_ms=elapsed,sequence_agreement=agreement))
            debug_record('candidate',dict(mode=mode,name=name,settings=settings,status=status,setup_seconds=time.perf_counter()-started,
                timing_samples_ms=values,validation=[dict(correct=correctness(a,r),tokens=a['tokens']) for a,r in zip(results,validation)]))
            if engine is not best: del engine
        except Exception as error:
            if 'illegal memory access' in str(error).lower() or 'device-side assert' in str(error).lower(): raise
            message=f'{type(error).__name__}: {error}'
            candidate_rows.append(dict(mode=mode,candidate=name,status='unavailable / failed check',validation_correct=None,
                                       validation_total=len(validation),mean_ms=None,sequence_agreement=None))
            debug_record('candidate_failure',dict(mode=mode,candidate=name,error=message))
            print(f'{name}: {message[:240]}',flush=True)
            engine=None
        plain=None; plain_dense=None; dense_trial=None
        gc.collect(); torch.cuda.empty_cache()
    selected[mode]=dict(engine=best,dense_engine=best_dense,settings=best_config,name=best_name if best is not None else 'current (no qualifying speedup)')
    print(f"Selected {mode}: {selected[mode]['name']}",flush=True)
print('Candidate selection (validation data only):')
candidate_table=pd.DataFrame(candidate_rows)
display(candidate_table[['mode','candidate','status','mean_ms','validation_correct']].rename(columns={'mean_ms':'Mean ms/question','validation_correct':f'Correct / {len(validation)}'}))
debug_record('selection',dict(candidates=candidate_rows,selected={m:dict(name=s['name'],settings=s['settings']) for m,s in selected.items()}))

Prepare the dense-head control with the selected transformer settings, then warm all paths. No module hooks are installed.

In [ ]:
dense_engines={mode:selected[mode]['dense_engine'] for mode in MODES}

def run(version,mode,arm,question,diagnostic=False):
    engine=(selected[mode]['engine'] if arm=='rank96' else dense_engines[mode]) if version=='optimized' else None
    if engine is None: return baseline(mode,arm,question,diagnostic)
    return engine.generate(question,mode,MAX_NEW_TOKENS[mode],diagnostic)

for version in ('current','optimized'):
    for mode in MODES:
        for arm in ('dense','rank96'):
            for question in warmup: run(version,mode,arm,question)
assert not any(m._forward_hooks or m._forward_pre_hooks for m in model.modules())
# One optional CUPTI trace, outside every timing/accuracy measurement.
try:
    with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU,torch.profiler.ProfilerActivity.CUDA]) as probe:
        run('optimized','explicit_cot','rank96',warmup[0])
    probe.export_chrome_trace(str(RUN_DIR/'optimized_gpu_trace.json'))
    print('Optional GPU kernel timeline saved: optimized_gpu_trace.json')
except Exception as error:
    debug_record('gpu_trace_unavailable',str(error))
    print('CUPTI trace unavailable; timing/accuracy remain enabled:',str(error)[:160])

**Main benchmark: no hooks, no per-layer events, no diagnostic graphs.**
Compilation, warmup and traces finish before timing. Both heads use their own outputs,
so generated lengths and accuracy are reported with latency. Each result is retained
individually; the displayed numbers are means.

In [ ]:
clean_samples=[]
rng=random.Random(SEED)
for repeat in range(TIMING_REPEATS):
    jobs=[(version,mode,arm,i) for version in ('current','optimized') for mode in MODES
          for arm in ('dense','rank96') for i in range(len(splits['timing']))]
    rng.shuffle(jobs)
    for version,mode,arm,i in jobs:
        question=splits['timing'][i]['question']
        sample=opt.timed(lambda:run(version,mode,arm,question),device)
        sample.update(version=version,arm=arm,question_id=i,repeat=repeat)
        clean_samples.append(sample)
        debug_record('clean_sample',sample)
    print(f'Clean timing {repeat+1}/{TIMING_REPEATS} complete.',flush=True)

def clean_mean(version,mode,arm,key='wall_ms'):
    group=[s[key] for s in clean_samples if s['version']==version and s['mode']==mode and s['arm']==arm]
    return sum(group)/len(group)
for mode,label in [('explicit_cot','Explicit'),('codi','CODI')]:
    for arm in ('dense','rank96'):
        before,after=(clean_mean(v,mode,arm) for v in ('current','optimized'))
        counts=[clean_mean(v,mode,arm,'visible_tokens') for v in ('current','optimized')]
        print(f'{label}, {arm}: {before:.2f} -> {after:.2f} ms/question ({before/after:.2f}x); visible tokens {counts[0]:.2f} -> {counts[1]:.2f}.')
    saved=[100*(1-clean_mean(v,mode,'rank96')/clean_mean(v,mode,'dense')) for v in ('current','optimized')]
    print(f'  Latency saved by head compression: {saved[0]:.1f}% before transformer changes; {saved[1]:.1f}% after (free-generation comparison).')

Four **coarse diagnostic** tables, using just two questions from the timing set,
separate from the clean means above. There are still no module hooks. Whole compiled
transformer calls are measured together; instrumenting their internal blocks would
break the optimization being measured. The earlier isolated table shows block substeps.

For CUDA Graphs, a separate diagnostic graph records stage events; the main graph stays
uninstrumented. If this PyTorch build cannot capture timing events, the notebook explicitly
labels the diagnostic fallback. Synchronization/measurement overhead is included in these
tables and must not be interpreted as production latency.

CODI overall = shared + latent + visible. The per-token denominators are visible output
tokens for Explicit/visible, six steps for latent, and latent + visible for CODI overall.

In [ ]:
diagnostic_samples=[]
for mode in MODES:
    engine=selected[mode]['engine']
    if engine is not None:
        engine.prepare_diagnostics()
        if engine.diagnostic_graph_error:
            print(f'{mode}: diagnostic graph events unavailable; using compiled calls without graph replay for diagnostics only.')
            debug_record('diagnostic_graph_fallback',dict(mode=mode,error=engine.diagnostic_graph_error))
for version in ('current','optimized'):
    for mode in MODES:
        for i,row in enumerate(splits['timing'][:DIAGNOSTIC_QUESTIONS]):
            sample=opt.timed(lambda:run(version,mode,'rank96',row['question'],diagnostic=True),device)
            sample.update(version=version,arm='rank96',question_id=i)
            # Clocks must not change the selected decoder's outputs.
            expected=next(s for s in clean_samples if s['version']==version and s['mode']==mode and s['arm']=='rank96' and s['question_id']==i)
            assert sample['tokens']==expected['tokens'],'Diagnostic path changed generated tokens'
            diagnostic_samples.append(sample)
            debug_record('coarse_diagnostic',sample)
tables={}
for per_token in (False,True):
    for version,label in [('current','Before transformer optimization'),('optimized','After transformer optimization')]:
        samples=[s for s in diagnostic_samples if s['version']==version]
        report=opt.coarse_table(samples,per_token)
        frame=pd.DataFrame([r for _,r in report],index=[name for name,_ in report],columns=runtime.COLUMNS)
        key=version+('_per_token' if per_token else '_per_question')
        tables[key]=frame
        print(label+' - '+('ms/token or latent step' if per_token else 'ms/question')+' (coarse diagnostic, rank96 head)')
        if not per_token:
            t=frame.iloc[0]
            shared=t['CODI overall']-t['CODI latent only']-t['CODI visible only']
            print(f"CODI: {t['CODI overall']:.3f} = {shared:.3f} shared + {t['CODI latent only']:.3f} latent + {t['CODI visible only']:.3f} visible.")
        with pd.option_context('display.float_format',lambda x:f'{x:.3f}','display.max_columns',4,'display.max_rows',None):
            display(frame)
save_json(RUN_DIR/'timing_tables.json',{key:frame.to_dict('split') for key,frame in tables.items()})
for mode,label in [('explicit_cot','Explicit'),('codi','CODI')]:
    for version in ('current','optimized'):
        group=[s for s in diagnostic_samples if s['version']==version and s['mode']==mode]
        share=100*sum(p['ms'] for s in group for p in s['parts'] if p['name']=='LM head + argmax')/sum(s['wall_ms'] for s in group)
        print(f'{label}, {version}: head + argmax {share:.1f}% of diagnostic time. Use clean dense/rank96 controls above for actual latency impact.')

Full GSM8K test accuracy for current/optimized transformer execution and both heads.
All 1,319 questions are evaluated once per combination, without hooks or clocks inside
the decoder. Quantized selections are clearly named above; test accuracy is never used
to choose the winner. This is the longest evaluation step.

In [ ]:
accuracy_results={}
for mode in MODES:
    for version in ('current','optimized'):
        for arm in ('dense','rank96'):
            correct=0; capped=0
            for i,row in enumerate(test_rows):
                result=run(version,mode,arm,row['question'])
                match=answers_match(result['text'],row['gold'])
                hit_cap=result['tokens'][-1]!=tokenizer.eos_token_id
                correct+=int(match); capped+=int(hit_cap)
                debug_record('accuracy_sample',dict(version=version,mode=mode,arm=arm,question_id=i,
                    text=result['text'],tokens=result['tokens'],gold=row['gold'],correct=match,hit_cap=hit_cap))
                if (i+1)%256==0: print(f'Accuracy {mode}/{version}/{arm}: {i+1}/{len(test_rows)}',flush=True)
            result=dict(correct=correct,total=len(test_rows),accuracy=correct/len(test_rows),capped=capped)
            accuracy_results[mode,version,arm]=result
            debug_record('accuracy_result',dict(mode=mode,version=version,arm=arm,**result))
for mode,label in [('explicit_cot','Explicit'),('codi','CODI')]:
    for arm in ('dense','rank96'):
        before,after=(accuracy_results[mode,v,arm] for v in ('current','optimized'))
        print(f"{label}, {arm}: {before['accuracy']:.2%} ({before['correct']}/{before['total']}) -> "
              f"{after['accuracy']:.2%} ({after['correct']}/{after['total']}); token-cap hits {before['capped']} -> {after['capped']}.")
print('Done. Main timings: clean_samples. Four diagnostic tables: tables. Accuracy: accuracy_results.')
print('All individual measurements, candidate failures and predictions:',DEBUG_PATH)

Optional debugging stays in the notebook: `clean_samples`, `diagnostic_samples`,
`substep_events`, `candidate_rows`, `accuracy_results`. The single compressed debug log
retains raw values. `optimized_gpu_trace.json`, when available, opens in Perfetto.

Optimization research behind the experiments:
[PyTorch GPT-fast](https://pytorch.org/blog/accelerating-generative-ai-2/),
[CUDA Graphs](https://pytorch.org/blog/accelerating-pytorch-with-cuda-graphs/),
[static caches](https://huggingface.co/docs/transformers/kv_cache).
The equal-MAC test is not proof of a fixed hardware tax; performance and quality can
improve, remain unchanged, or regress. If no faster candidate passes validation, the
notebook says so and retains the current decoder.